In [30]:
from google import genai

client = genai.Client(api_key="GEMINI_API_KEY"])

In [ ]:
# config={"temperature": 0.0}  # 거의 결정적인 답
# config={"temperature": 1.5}  # 더 창의적/무작위

In [19]:
response = client.models.generate_content(
    model="gemini-3.8-flash",
    contents="안녕하세요, 자기소개 해주세요."
)
print(response.text)

안녕하세요! 저는 Google에서 개발한 대형 언어 모델, **Gemini(제미나이)**입니다.

저는 여러분과 자연스럽게 대화하며 다양한 작업을 도와드릴 수 있는 AI 어시스턴트입니다. 주로 다음과 같은 일들을 잘할 수 있어요:

* **궁금증 해결:** 일상적인 질문부터 학술적·전문적인 지식까지 다양한 정보를 찾아 알기 쉽게 설명해 드립니다.
* **글쓰기 및 교정:** 이메일, 보고서, 블로그 글, 창작물(시, 이야기) 등을 작성하거나 문맥에 맞게 다듬어 드립니다.
* **번역 및 외국어 학습:** 다양한 언어로의 번역과 자연스러운 표현 학습을 돕습니다.
* **정보 요약 및 분석:** 긴 문서나 복잡한 글의 핵심 내용을 빠르게 파악할 수 있도록 요약해 드립니다.
* **코딩 지원:** 프로그래밍 코드 작성, 오류 수정(디버깅), 알고리즘 설명 등을 지원합니다.

궁금한 점이 있거나 도움이 필요한 작업이 있으시다면 언제든 편하게 말씀해 주세요. 오늘 어떤 것을 도와드릴까요?


In [22]:
response = client.models.generate_content(
   model="gemini-3.8-flash",
   config={"system_instruction": "너는 친절한 파이썬 튜터야. 초보자 눈높이에 맞춰 설명해."},
   contents="리스트 컴프리헨션이 뭐야?"
)

In [24]:
response = client.models.generate_content(
   model="gemini-3.8-flash",
   config={"response_mime_type": "application/json", "temperature": 1.5},  # 더 창의적/무작위
   contents="사과, 바나나, 포도를 JSON 배열로 만들어줘"
)
print(response.text)

["사과", "바나나", "포도"]


In [25]:
print(response.usage_metadata)

cache_tokens_details=None cached_content_token_count=None candidates_token_count=13 candidates_tokens_details=None prompt_token_count=15 prompt_tokens_details=[ModalityTokenCount(
  modality=<MediaModality.TEXT: 'TEXT'>,
  token_count=15
)] thoughts_token_count=164 tool_use_prompt_token_count=None tool_use_prompt_tokens_details=None total_token_count=192 traffic_type=None


RAG

In [27]:
documents = [
    "연차 휴가는 입사일 기준으로 매년 15일이 발생하며, 1년 미만 근무자는 매월 1일씩 적립됩니다. 미사용 연차는 다음 해로 이월되지 않으며 연말에 소멸됩니다.",
    "재택근무는 주 2회까지 가능하며, 팀장의 사전 승인이 필요합니다. 재택근무 시에도 핵심 업무 시간(오전 10시~오후 4시)에는 접속 상태를 유지해야 합니다.",
    "출장비는 교통비, 숙박비, 식비로 구분되며, 국내 출장 시 1일 식비 한도는 3만원입니다. 영수증은 출장 종료 후 7일 이내에 제출해야 정산됩니다.",
    "신입사원은 입사 후 3개월간 수습 기간을 거치며, 이 기간 동안 급여의 90%가 지급됩니다. 수습 평가는 직속 상사가 진행하며 인사팀에 결과를 제출합니다.",
    "사내 동호회 활동비는 분기별로 인당 5만원까지 지원되며, 10인 이상 활동하는 동호회에 한해 신청 가능합니다. 신청은 복지포털을 통해 진행합니다.",
    "퇴사 시에는 최소 30일 전에 팀장과 인사팀에 통보해야 하며, 인수인계 문서를 작성하여 제출해야 합니다. 미사용 연차는 금전으로 정산됩니다.",
]

In [29]:
import numpy as np

def get_embedding(text):
    result = client.models.embed_content(
        model="gemini-embedding-2",
        contents=text
    )
    return np.array(result.embeddings[0].values)

In [31]:
doc_embeddings = [get_embedding(doc) for doc in documents]
print(f"문서 {len(doc_embeddings)}개 임베딩 완료, 차원: {len(doc_embeddings[0])}")

문서 6개 임베딩 완료, 차원: 3072


In [32]:
def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

def search(query, top_k=2):
    query_embedding = get_embedding(query)
    similarities = [cosine_similarity(query_embedding, doc_emb) for doc_emb in doc_embeddings]
    top_indices = np.argsort(similarities)[::-1][:top_k]
    return [(documents[i], similarities[i]) for i in top_indices]

In [33]:
results = search("휴가 며칠 쓸 수 있어?")
for doc, score in results:
    print(f"[{score:.3f}] {doc}")

[0.643] 연차 휴가는 입사일 기준으로 매년 15일이 발생하며, 1년 미만 근무자는 매월 1일씩 적립됩니다. 미사용 연차는 다음 해로 이월되지 않으며 연말에 소멸됩니다.
[0.569] 재택근무는 주 2회까지 가능하며, 팀장의 사전 승인이 필요합니다. 재택근무 시에도 핵심 업무 시간(오전 10시~오후 4시)에는 접속 상태를 유지해야 합니다.


In [34]:
def rag_answer(query):
    relevant_docs = search(query, top_k=2)
    context = "\n\n".join([doc for doc, score in relevant_docs])
    
    prompt = f"""다음 문서를 참고해서 질문에 답해줘. 문서에 없는 내용은 지어내지 말고 "문서에서 찾을 수 없습니다"라고 답해.

[참고 문서]
{context}

[질문]
{query}
"""
    response = client.models.generate_content(
        model="gemini-3.8-flash",
        config={"temperature": 0.1},
        contents=prompt
    )
    return response.text

In [37]:
print(rag_answer("연차는 며칠이고 이월 되나요?"))

연차 휴가는 입사일 기준으로 매년 15일이 발생(1년 미만 근무자는 매월 1일씩 적립)하며, 다음 해로 이월되지 않고 연말에 소멸됩니다.


In [38]:
print(rag_answer("회식비는 얼마까지 지원돼?"))  # 문서에 없는 내용 → "찾을 수 없습니다" 나오는지 확인

문서에서 찾을 수 없습니다.


In [39]:
results = search("동호회 활동비 얼마까지 돼?")
for doc, score in results:
    print(f"[{score:.3f}] {doc}")

[0.758] 사내 동호회 활동비는 분기별로 인당 5만원까지 지원되며, 10인 이상 활동하는 동호회에 한해 신청 가능합니다. 신청은 복지포털을 통해 진행합니다.
[0.612] 출장비는 교통비, 숙박비, 식비로 구분되며, 국내 출장 시 1일 식비 한도는 3만원입니다. 영수증은 출장 종료 후 7일 이내에 제출해야 정산됩니다.


In [41]:
print(rag_answer("퇴사할 때 연차는 어떻게 처리돼?"))

퇴사 시 미사용 연차는 금전으로 정산됩니다.


In [42]:
def search_with_threshold(query, top_k=2, threshold=0.5):
    query_embedding = get_embedding(query)
    similarities = [cosine_similarity(query_embedding, doc_emb) for doc_emb in doc_embeddings]
    top_indices = np.argsort(similarities)[::-1][:top_k]
    results = [(documents[i], similarities[i]) for i in top_indices if similarities[i] >= threshold]
    return results

In [45]:
for t in [0.3, 0.5, 0.6, 0.65, 0.7]:
    results = search_with_threshold("오늘 점심 뭐 먹지?", threshold=t)
    print(f"threshold={t}: {len(results)}개 결과")

threshold=0.3: 2개 결과
threshold=0.5: 1개 결과
threshold=0.6: 0개 결과
threshold=0.65: 0개 결과
threshold=0.7: 0개 결과


In [46]:
results = search_with_threshold("연차 며칠 쓸 수 있어?", threshold=0.65)
print(results)

[('연차 휴가는 입사일 기준으로 매년 15일이 발생하며, 1년 미만 근무자는 매월 1일씩 적립됩니다. 미사용 연차는 다음 해로 이월되지 않으며 연말에 소멸됩니다.', np.float64(0.6796577515754656))]
